In [24]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import prism
import xarray as xr

from imagematerials import buildings as bld
from imagematerials import vehicles as vhc
from imagematerials.concepts import create_building_graph, create_vehicle_graph
from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import GenericMaterials, GenericStocks, MaterialIntensities
from imagematerials.util import (
    export_to_netcdf,
    import_from_netcdf,
    read_circular_economy_config,
    read_climate_policy_config,
    rebroadcast_prep_data,
)
from imagematerials.preprocessing import get_preprocessing_data


In [ ]:
vehicle_sector = get_preprocessing_data("vehicles", Path("..", "data", "raw"), cache="prep_vema_new.nc")

In [11]:
buildings_sector = get_preprocessing_data("buildings", Path("..", "data", "raw"), cache="prep_buildings_new.nc")

In [12]:
sectors = [buildings_sector, vehicle_sector]
ns_coordinates = {
    "Region": sectors[0].coordinates["Region"],
    "material": list(set(sectors[0].coordinates["material"] + sectors[1].coordinates["material"]))
}
ns_combined = Sector("combi", {}, coordinates=ns_coordinates)
sectors.append(ns_combined)

In [ ]:
# # Define the complete timeline, including historic tail
# # time_start = prep_data["stocks"].coords["Time"].min().values
# time_start = 1960
# complete_timeline = prism.Timeline(time_start, 2060, 1)
# simulation_timeline = prism.Timeline(1970, 2060, 1)

In [13]:
# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2100, 1)
simulation_timeline = prism.Timeline(1970, 2100, 1)

In [14]:
REGION = prism.Dimension("Region")
MATERIAL_TYPE = prism.Dimension("material")

@prism.interface
class SumMaterials(prism.Model):
    Region: prism.Coords[REGION]
    material: prism.Coords[MATERIAL_TYPE]

    input_data: tuple[str] = ("inflow_materials", )
    output_data: tuple[str] = ("summed_inflow_materials", )



    summed_inflow_materials: prism.TimeVariable[REGION, MATERIAL_TYPE, "count"] = prism.export()

    def compute_initial_values(self, time: prism.Timeline):
        pass

    def compute_values(self, time: prism.Time, inflow_materials):
        t, dt = time.t, time.dt
        self.summed_inflow_materials[t].loc[:] = 0
        for inflow in inflow_materials:
            self.summed_inflow_materials[t].loc[{"material": inflow[t].coords["material"]}] += inflow[t].sum("Type")

In [33]:
gompertz = pd.read_csv(
    Path("..", "data", "raw", "rest-of", "metals", "steel_region_model_coefs.csv"),
    index_col=0,
    header=[0],
)

array_list = []
for name in ["a", "b", "c"]:
    array_list.append(xr.DataArray(gompertz[name], dims=("Region",), coords={"Region": gompertz.index}))
xr_gompertz = xr.concat(array_list, dim="params")
xr_gompertz.coords["params"] = ["a", "b", "c"]
xr_gompertz

<xarray.DataArray 'a' (params: 3, Region: 26)> Size: 624B
array([[3.83978468e-01, 2.30596900e-01, 3.05671831e-01, 3.05671831e-01,
        3.05671831e-01, 3.05671831e-01, 3.05671831e-01, 3.05671831e-01,
        3.05671831e-01, 3.05671831e-01, 2.30596900e-01, 3.05671831e-01,
        3.05671831e-01, 3.05671831e-01, 3.05671831e-01, 6.51230934e-01,
        3.05671831e-01, 3.05671831e-01, 6.51230934e-01, 6.51230934e-01,
        3.05671831e-01, 3.05671831e-01, 3.83978468e-01, 2.30596900e-01,
        3.05671831e-01, 3.05671831e-01],
       [3.68985995e+06, 3.37997590e+06, 3.35772799e+00, 3.35772799e+00,
        3.35772799e+00, 3.35772799e+00, 3.35772799e+00, 3.35772799e+00,
        3.35772799e+00, 3.35772799e+00, 3.37997590e+06, 3.35772799e+00,
        3.35772799e+00, 3.35772799e+00, 3.35772799e+00, 3.65178272e+00,
        3.35772799e+00, 3.35772799e+00, 3.65178272e+00, 3.65178272e+00,
        3.35772799e+00, 3.35772799e+00, 3.68985995e+06, 3.37997590e+06,
        3.35772799e+00, 3.35772799e+00],
       [2.71240824e+05, 2.58826364e+05, 2.00375048e-01, 2.00375048e-01,
        2.00375048e-01, 2.00375048e-01, 2.00375048e-01, 2.00375048e-01,
        2.00375048e-01, 2.00375048e-01, 2.58826364e+05, 2.00375048e-01,
        2.00375048e-01, 2.00375048e-01, 2.00375048e-01, 1.82701983e-01,
        2.00375048e-01, 2.00375048e-01, 1.82701983e-01, 1.82701983e-01,
        2.00375048e-01, 2.00375048e-01, 2.71240824e+05, 2.58826364e+05,
        2.00375048e-01, 2.00375048e-01]])
Coordinates:
  * Region   (Region) int64 208B 1 2 3 4 5 6 7 8 9 ... 19 20 21 22 23 24 25 26
  * params   (params) <U1 12B 'a' 'b' 'c'

In [17]:
# Rest of prism model 

@prism.interface
class RestOfGompertz(prism.Model):
    Region: prism.Coords[REGION]
    material: prism.Coords[MATERIAL_TYPE]

    input_data: tuple[str] = ("gdp_cap", "rest_of_gompertz_params")
    output_data: tuple[str] = ("inflow_materials_other")

    inflow_materials_other: prism.TimeVariable[REGION, MATERIAL_TYPE, "count"] = prism.export()

    def compute_initial_values(self, time: prism.Timeline):
        pass

    def compute_values(self, time: prism.Timeline, gdp_cap, rest_of_gompertz_params):



SyntaxError: incomplete input (992640325.py, line 17)

In [ ]:
factory = ModelFactory(
    sectors, complete_timeline
    ).add(GenericStocks, ["bld", "vhc"]
    ).add(GenericMaterials,  "vhc"
    ).add(MaterialIntensities, "bld",
    ).add(SumMaterials, "combi", input_sources={"inflow_materials": ["vhc", "bld"]}
    )
model = factory.finish()

In [ ]:
import warnings
with warnings.catch_warnings():
    warnings.filterwarnings("ignore")
    model.simulate(simulation_timeline)

KeyError: "not all values found in index 'Time'. Try setting the `method` keyword argument (example: method='nearest')."

In [ ]:
model.combi["summed_inflow_materials"][2020]

<xarray.DataArray (Region: 26, material: 18)> Size: 4kB
<Quantity([[1.30922932e+07 5.66603540e+00 3.28308921e+00 0.00000000e+00
  0.00000000e+00 1.49737285e+08 1.96375743e+09 0.00000000e+00
  2.79774182e-02 2.02994289e+08 0.00000000e+00 9.38338280e+00
  1.36347205e+07 0.00000000e+00 1.26374013e+08 0.00000000e+00
  1.39101217e+05 0.00000000e+00]
 [4.14604060e+08 9.84437588e+01 7.79989420e+01 0.00000000e+00
  0.00000000e+00 4.33461551e+09 5.79780759e+10 0.00000000e+00
  5.96740596e-01 5.99420719e+09 0.00000000e+00 1.80645421e+02
  3.35218446e+08 0.00000000e+00 3.19192782e+09 0.00000000e+00
  7.15050299e+06 0.00000000e+00]
 [2.42232673e+07 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 2.64967308e+08 3.75284699e+09 0.00000000e+00
  0.00000000e+00 5.24457917e+08 0.00000000e+00 0.00000000e+00
  6.07693585e+07 0.00000000e+00 5.02569329e+08 0.00000000e+00
  8.84881914e+04 0.00000000e+00]
 [6.36368027e+07 0.00000000e+00 0.00000000e+00 0.00000000e+00
  0.00000000e+00 7.15928620e+08 9.29185347e+09 0.00000000e+00
  0.00000000e+00 1.19610954e+09 0.00000000e+00 0.00000000e+00
  6.40645024e+07 0.00000000e+00 5.06532244e+08 0.00000000e+00
  1.84993473e+05 0.00000000e+00]
...
 [5.72320060e+08 7.86179757e+00 4.55538678e+00 0.00000000e+00
  0.00000000e+00 6.03407557e+08 1.00358341e+10 1.09127400e+07
  3.88195243e-02 1.85917944e+09 0.00000000e+00 1.30197309e+01
  3.70192220e+08 2.70380373e+05 2.10350798e+09 5.11442898e+07
  4.38639679e+07 0.00000000e+00]
 [1.36523908e+07 5.55665284e+01 3.35888348e+01 0.00000000e+00
  0.00000000e+00 1.58278340e+08 2.24254284e+09 0.00000000e+00
  2.89043455e-01 3.24018582e+08 0.00000000e+00 9.46266344e+01
  2.68241954e+07 0.00000000e+00 2.05966950e+08 0.00000000e+00
  8.49896794e+05 0.00000000e+00]
 [5.75686505e+07 7.81797671e+00 4.52177209e+00 0.00000000e+00
  0.00000000e+00 6.12050710e+08 9.31103304e+09 0.00000000e+00
  3.96996814e-02 1.31986813e+09 0.00000000e+00 1.30680199e+01
  2.10656877e+08 0.00000000e+00 1.84065340e+09 0.00000000e+00
  7.38676047e+05 0.00000000e+00]
 [8.36110182e+07 5.07162686e+00 2.93490913e+00 0.00000000e+00
  0.00000000e+00 2.34231748e+08 3.72389144e+09 1.30740003e+06
  2.55437525e-02 5.91491790e+08 0.00000000e+00 8.45425823e+00
  1.07187945e+08 3.23929009e+04 7.79693270e+08 6.12733791e+06
  5.41319777e+06 0.00000000e+00]], 'count')>
Coordinates:
  * Region    (Region) <U2 208B '1' '2' '3' '4' '5' ... '22' '23' '24' '25' '26'
  * material  (material) <U9 648B 'Cu' 'Brick' 'Cement' ... 'Pb' 'Wood' 'Mn'